In [1]:
!pip --quiet install ijson
!pip --quiet install weaviate-client
!pip --quiet install torch
!pip --quiet install transformers
!pip --quiet install bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.5/114.5 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.4/330.4 kB 7.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.9/5.9 MB 80.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 82.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 319.7/319.7 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 3.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu12 24.4.1 req

In [2]:
import json # For reading json file
import ijson # For reading json file
import pandas as pd
from pathlib import Path
import numpy as np
import wandb
import weaviate
from weaviate.classes.init import Auth
from weaviate.util import generate_uuid5
import os
import weaviate.classes.config as wc

from transformers import AutoTokenizer, AutoModelForQuestionAnswering, Trainer, TrainingArguments, PreTrainedModel, PreTrainedTokenizerFast, AdamW,DataCollatorForLanguageModeling

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    weaviate_url = user_secrets.get_secret("WEAVIATE_URL")
    weaviate_api_key = user_secrets.get_secret("WEAVIATE_API_KEY")
    # Connect to Weaviate Cloud
    client = weaviate.connect_to_weaviate_cloud(
        cluster_url=weaviate_url,
        auth_credentials=Auth.api_key(weaviate_api_key),
    )
    anony=None
    print(client.is_ready())
except:
    anony = "must"
    

True


In [3]:
import wandb

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    api_key = user_secrets.get_secret("wandb_api")
    anony = None
except:
    anony = "must"
    print('If you want to use your W&B account, go to Add-ons -> Secrets and provide your W&B access token. Use the Label name as wandb_api. \nGet your W&B access token from here: https://wandb.ai/authorize')

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [4]:
!wandb login $api_key

/usr/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


In [5]:
with open('/kaggle/input/bioqav2024/BioASQ-training12b/BioASQ-training12b/training12b_new.json') as f:
    data = json.load(f)

In [6]:
# Use pd.json_normalize to convert the JSON to a DataFrame
df = pd.json_normalize(data['questions'],
                     meta=['body','documents','ideal_answer','exact_answer','concepts','type','id','snippets' ])

In [7]:
# if column exact_answer is Nan get value from column ideal_answer
df['exact_answer'] = df['exact_answer'].fillna(df['ideal_answer'])
df = df.rename(columns={'body': 'question'})

In [8]:
df

,question,documents,ideal_answer,concepts,type,id,snippets,triples,exact_answer,_body,_type
0,Is Hirschsprung disease a mendelian or a multi...,"[http://www.ncbi.nlm.nih.gov/pubmed/15858239, ...","[Coding sequence mutations in RET, GDNF, EDNRB...",[http://www.disease-ontology.org/api/metadata/...,summary,55031181e9bde69634000014,"[{'offsetInBeginSection': 131, 'offsetInEndSec...",NaN,"[Coding sequence mutations in RET, GDNF, EDNRB...",NaN,NaN
1,List signaling molecules (ligands) that intera...,"[http://www.ncbi.nlm.nih.gov/pubmed/23959273, ...",[The 7 known EGFR ligands are: epidermal grow...,[http://amigo.geneontology.org/cgi-bin/amigo/t...,list,55046d5ff8aee20f27000007,"[{'offsetInBeginSection': 1085, 'offsetInEndSe...",[{'p': 'http://purl.uniprot.org/core/encodedBy...,"[[epidermal growth factor], [betacellulin], [e...",NaN,NaN
2,Is the protein Papilin secreted?,"[http://www.ncbi.nlm.nih.gov/pubmed/3320045, h...","[Yes, papilin is a secreted protein]",NaN,yesno,54e25eaaae9738404b000017,"[{'offsetInBeginSection': 1085, 'offsetInEndSe...",NaN,yes,NaN,NaN
3,Are long non coding RNAs spliced?,"[http://www.ncbi.nlm.nih.gov/pubmed/22955988, ...",[Long non coding RNAs appear to be spliced thr...,[http://www.nlm.nih.gov/cgi/mesh/2014/MB_cgi?f...,yesno,535d292a9a4572de6f000003,"[{'offsetInBeginSection': 546, 'offsetInEndSec...",NaN,yes,NaN,NaN
4,Is RANKL secreted from the cells?,"[http://www.ncbi.nlm.nih.gov/pubmed/22948539, ...",[Receptor activator of nuclear factor κB ligan...,"[http://www.uniprot.org/uniprot/TNF11_RAT, htt...",yesno,55262a9787ecba3764000009,"[{'offsetInBeginSection': 114, 'offsetInEndSec...",NaN,yes,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5044,What is telegenetics?,[http://www.ncbi.nlm.nih.gov/pubmed/33817891],[Telegenetics involves the use of technology (...,NaN,summary,6415babe690f196b5100000d,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN,[Telegenetics involves the use of technology (...,NaN,NaN
5045,What is the mechanism of action of Mitapivat?,"[http://www.ncbi.nlm.nih.gov/pubmed/35576529, ...","[Mitapivat, an oral activator of pyruvate kina...",NaN,summary,63f04546f36125a426000024,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN,"[Mitapivat, an oral activator of pyruvate kina...",NaN,NaN
5046,Do cells undergoing necroptosis show disruptio...,"[http://www.ncbi.nlm.nih.gov/pubmed/33848465, ...",[Necroptosis is a form of caspase-independent ...,NaN,yesno,641357bc201352f04a000039,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN,yes,NaN,NaN
5047,What is the definition of dermatillomania?,"[http://www.ncbi.nlm.nih.gov/pubmed/33808008, ...",[Dermatillomania is a condition that leads to ...,NaN,summary,6414c3f3690f196b51000005,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN,[Dermatillomania is a condition that leads to ...,NaN,NaN


In [9]:
# Check unique value in _body and _type columns
print(df['_body'].unique())
print(df['_type'].unique())

[nan 'What does Retapamulin treat?' '"DPM1001"'
 'What is the effect of the venom of the cone snail, Conus tulip?'
 'Which are the main genes of formative pluripotency?'
 'To what chromosome would the MKKS gene for McKusick-Kaufman(AKA Kaufman-McKusick) syndrome be found?'
 'Which kinase does ??? inhibit?'
 'Is autism thought to be related to the gene for vasopressin?'
 'What are the symptoms of an incidental durotomy (ID)' 'epiregulin'
 'Where does B type Natriuretic Protein, BNP usually originate from?'
 'What is RT001?'
 'Which one was the first fragment-derived drug to be approved by the US Federal Drug Administration (FDA)?'
 'Is OXLUMO (lumasiran) used for the treatment of primary hyperoxaluria'
 'What process does orexin/hypocretin neurons regulate']
[nan 'yesno' 'summary' 'list' 'factoid']


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [10]:
print(df['question'].unique().all())

What disease can be treated with Glofitamab?


In [11]:
#drop _body and _type columns
df = df.drop(columns=['_body', '_type'])

In [12]:
# plot body and answer
print(df['question'][0])
print(df['ideal_answer'][0])

Is Hirschsprung disease a mendelian or a multifactorial disorder?
["Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease. The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model."]


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [13]:
# Get question, documents, snippets and ideal answera and save to train dataframe
train_df = df[['question', 'documents', 'snippets', 'ideal_answer']]

In [14]:
# Check if exist Nan value
train_df.isnull().sum()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


question        0
documents       0
snippets        0
ideal_answer    0
dtype: int64

In [15]:
# Plots snippets
train_df['snippets'][0]

[{'offsetInBeginSection': 131,
  'offsetInEndSection': 358,
  'text': 'Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes',
  'beginSection': 'abstract',
  'document': 'http://www.ncbi.nlm.nih.gov/pubmed/15829955',
  'endSection': 'abstract'},
 {'offsetInBeginSection': 554,
  'offsetInEndSection': 992,
  'text': "In this study, we review the identification of genes and loci involved in the non-syndromic common form and syndromic Mendelian forms of Hirschsprung's disease. The majority of the identified genes are related to Mendelian syndromic forms of Hirschsprung's disease. The non-Mendelian inheritance of sporadic non-syndromic Hirschsprung's disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model",
  'beginSection': 'abstract',
  'document': 'http://www.ncbi.nlm.ni

In [16]:
data = pd.DataFrame(train_df['snippets'][0], columns=['offsetInBeginSection','offsetInEndSection','text','document','beginSection'])
# get document column last number till "/"
data['document'] = data['document'].astype(str).str.split('/').str[-1]
data = pd.concat([data,  pd.DataFrame(train_df.iloc[0]).T], axis = 1)
data['question'] = data['question'].fillna(str(train_df['question'][0]))
data['ideal_answer'] = data['ideal_answer'].fillna(str(train_df['ideal_answer'][0]))
data = data.drop(columns=['snippets','documents'])

In [17]:
data

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


,offsetInBeginSection,offsetInEndSection,text,document,beginSection,question,ideal_answer
0,131,358,Hirschsprung disease (HSCR) is a multifactoria...,15829955,abstract,Is Hirschsprung disease a mendelian or a multi...,"[Coding sequence mutations in RET, GDNF, EDNRB..."
1,554,992,"In this study, we review the identification of...",15617541,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
2,397,939,"Coding sequence mutations in e.g. RET, GDNF, E...",12239580,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
3,941,1279,For almost all of the identified HSCR genes in...,12239580,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
4,129,358,Hirschsprung disease (HSCR) is a multifactori...,15829955,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
5,851,1007,The inheritance of Hirschsprung disease is ge...,6650562,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
6,131,359,Hirschsprung disease (HSCR) is a multifactoria...,15829955,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
7,0,131,"Differential contributions of rare and common,...",20598273,title,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
8,0,210,BACKGROUND: RET is the major gene associated t...,21995290,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."
9,151,376,In the etiology of Hirschsprung disease variou...,15858239,abstract,Is Hirschsprung disease a mendelian or a multi...,"[""Coding sequence mutations in RET, GDNF, EDNR..."


In [18]:
temp = pd.Series(train_df["ideal_answer"].apply(lambda x: ''.join(x)), name = 'extracted_answer')
print(type(temp))
temp

<class 'pandas.core.series.Series'>


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


0       Coding sequence mutations in RET, GDNF, EDNRB,...
1       The 7 known EGFR ligands  are: epidermal growt...
2                     Yes,  papilin is a secreted protein
3       Long non coding RNAs appear to be spliced thro...
4       Receptor activator of nuclear factor κB ligand...
                              ...                        
5044    Telegenetics involves the use of technology (g...
5045    Mitapivat, an oral activator of pyruvate kinas...
5046    Necroptosis is a form of caspase-independent p...
5047    Dermatillomania is a condition that leads to r...
5048    Glofitamab is being tested for treatment of DL...
Name: extracted_answer, Length: 5049, dtype: object

In [19]:
# train_df.add(temp,axis = 'columns')
temp_df =  train_df.merge(temp, left_index = True, right_index = True)
temp_df

,question,documents,snippets,ideal_answer,extracted_answer
0,Is Hirschsprung disease a mendelian or a multi...,"[http://www.ncbi.nlm.nih.gov/pubmed/15858239, ...","[{'offsetInBeginSection': 131, 'offsetInEndSec...","[Coding sequence mutations in RET, GDNF, EDNRB...","Coding sequence mutations in RET, GDNF, EDNRB,..."
1,List signaling molecules (ligands) that intera...,"[http://www.ncbi.nlm.nih.gov/pubmed/23959273, ...","[{'offsetInBeginSection': 1085, 'offsetInEndSe...",[The 7 known EGFR ligands are: epidermal grow...,The 7 known EGFR ligands are: epidermal growt...
2,Is the protein Papilin secreted?,"[http://www.ncbi.nlm.nih.gov/pubmed/3320045, h...","[{'offsetInBeginSection': 1085, 'offsetInEndSe...","[Yes, papilin is a secreted protein]","Yes, papilin is a secreted protein"
3,Are long non coding RNAs spliced?,"[http://www.ncbi.nlm.nih.gov/pubmed/22955988, ...","[{'offsetInBeginSection': 546, 'offsetInEndSec...",[Long non coding RNAs appear to be spliced thr...,Long non coding RNAs appear to be spliced thro...
4,Is RANKL secreted from the cells?,"[http://www.ncbi.nlm.nih.gov/pubmed/22948539, ...","[{'offsetInBeginSection': 114, 'offsetInEndSec...",[Receptor activator of nuclear factor κB ligan...,Receptor activator of nuclear factor κB ligand...
...,...,...,...,...,...
5044,What is telegenetics?,[http://www.ncbi.nlm.nih.gov/pubmed/33817891],"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",[Telegenetics involves the use of technology (...,Telegenetics involves the use of technology (g...
5045,What is the mechanism of action of Mitapivat?,"[http://www.ncbi.nlm.nih.gov/pubmed/35576529, ...","[{'offsetInBeginSection': 0, 'offsetInEndSecti...","[Mitapivat, an oral activator of pyruvate kina...","Mitapivat, an oral activator of pyruvate kinas..."
5046,Do cells undergoing necroptosis show disruptio...,"[http://www.ncbi.nlm.nih.gov/pubmed/33848465, ...","[{'offsetInBeginSection': 0, 'offsetInEndSecti...",[Necroptosis is a form of caspase-independent ...,Necroptosis is a form of caspase-independent p...
5047,What is the definition of dermatillomania?,"[http://www.ncbi.nlm.nih.gov/pubmed/33808008, ...","[{'offsetInBeginSection': 0, 'offsetInEndSecti...",[Dermatillomania is a condition that leads to ...,Dermatillomania is a condition that leads to r...


In [20]:
# Check if still exist "[" in the start of text and "]" in the end of text
for index, row in temp_df.iterrows():
    # if "[" or "]" in row["temp"]:
    if "[" in row["extracted_answer"][0] and "]" in row["extracted_answer"]:
        # row["temp"] = row["temp"].replace("[","")
        # row["temp"] = row["temp"].replace("]","")
        print(row["extracted_answer"])
# The result look okiee to me

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


[1,7]naphthyridine-3-carbonitriles and quinoline-3-carbonitriles were the first Tumor Progression Loci-2 (Tpl2) kinase inhibitors. 4-alkylamino-[1,7]naphthyridine-3-carbonitriles are also known to inhibit Tpl2 function as well as quinoline-3-carbonitrile derivatives, thieno[3,2-d]pyrimidines and 2,4-disubstituted thieno[2,3-c]pyridines, indazoles, 4-Alkylamino-[1,7]naphthyridine-3-carbonitriles and generally molecules belonging to the wide categories of quinoline-3-carbonitriles, indazoles and thieno-pyrimidines.Honokiol
Thieno[3,2-d]pyrimidines and thieno[2,3-c]pyridine
Quinoline-3-carbonitrile derivatives (8-halo-4-(3-chloro-4-fluoro-phenylamino)-6-[(1H-[1,2,3]triazol-4-ylmethyl)-amino]-quinoline-3-carbonitriles; 8-bromo-4-(3-chloro-4-fluorophenylamino)-6-[(1-methyl-1H-imidazol-4-yl)methylamino]quinoline-3-carbonitrile; 4-alkylamino-[1,7]naphthyridine-3-carbonitrile; 1,7-naphthyridine-3-carbonitriles)
Indazoles
Luteolin
1,7-naphtyridine-3-carbonitrile
[' Numerous agents including cyt

In [21]:
# def transform_snippets_to_rows(train_df):
#     """
#     Chuyển đổi cột snippets thành nhiều hàng và nhân bản các cột khác

#     Parameters:
#     -----------
#     train_df : DataFrame
#         DataFrame gốc có cột snippets cần được chuyển đổi

#     Returns:
#     --------
#     DataFrame
#         DataFrame đã được chuyển đổi với mỗi snippet thành một hàng riêng
#     """

#     # Khởi tạo list để lưu các DataFrame con
#     transformed_dfs = []

#     # Xử lý từng hàng trong train_df
#     for idx in range(len(train_df)):
#         # Tạo DataFrame từ snippets của hàng hiện tại
#         snippet_df = pd.DataFrame(
#             train_df['snippets'].iloc[idx],
#             columns=['offsetInBeginSection', 'offsetInEndSection', 'text', 'document', 'beginSection']
#         )

#         # Xử lý cột document
#         snippet_df['document'] = snippet_df['document'].astype(str).str.split('/').str[-1]

#         # Thêm các cột khác từ hàng gốc
#         other_cols = pd.DataFrame(train_df.iloc[idx]).T
#         snippet_df = pd.concat([snippet_df, other_cols], axis=1)

#         # Điền tất cả giá trị NaN bằng giá trị từ hàng đầu tiên của snippet_df
#         # Check if the value is a list before filling NaNs
#         for column in snippet_df.columns:
#             # Exclude the text
#             if column == 'text' or column == 'offsetInBeginSection' or column == 'offsetInEndSection':
#                 continue
#             if snippet_df[column].isnull().any():
#                 fill_value = snippet_df[column].iloc[0]
#                 # If the fill value is a list, use an empty list instead
#                 if isinstance(fill_value, list):
#                     fill_value = [] if fill_value == float('nan') else fill_value[0] # Handle NaN in fill_value
#                 snippet_df[column] = snippet_df[column].fillna(fill_value)
#         # Điền các giá trị cho question và ideal_answer
#         snippet_df['question'] = snippet_df['question'].fillna(str(train_df['question'].iloc[idx]))
#         snippet_df['ideal_answer'] = snippet_df['ideal_answer'].fillna(str(train_df['ideal_answer'].iloc[idx]))
#         # Loại bỏ các hàng có cả 3 giá trị null
#         null_mask = snippet_df[['offsetInBeginSection', 'offsetInEndSection', 'text']].isnull().all(axis=1)
#         snippet_df = snippet_df[~null_mask]
#         # Thêm vào list các DataFrame đã xử lý
#         transformed_dfs.append(snippet_df)

#     # Ghép tất cả các DataFrame con
#     result_df = pd.concat(transformed_dfs, axis=0, ignore_index=True)

#     # Loại bỏ các cột không cần thiết
#     result_df = result_df.drop(columns=['snippets', 'documents'])

#     return result_df

def transform_snippets_to_rows(train_df):
    """
    Chuyển đổi cột snippets thành nhiều hàng và nhân bản các cột khác

    Parameters:
    -----------
    train_df : DataFrame
        DataFrame gốc có cột snippets cần được chuyển đổi

    Returns:
    --------
    DataFrame
        DataFrame đã được chuyển đổi với mỗi snippet thành một hàng riêng
    """

    # Khởi tạo list để lưu các DataFrame con
    transformed_dfs = []

    # Xử lý từng hàng trong train_df
    for idx in range(len(train_df)):
        try:
            snippets = train_df['snippets'].iloc[idx]
            
            # Kiểm tra nếu snippets là None hoặc rỗng
            if snippets is None or len(snippets) == 0:
                continue
                
            # Kiểm tra và chuyển đổi dữ liệu thành list nếu cần
            if isinstance(snippets, dict):
                snippets = [snippets]
            elif not isinstance(snippets, list):
                continue

            # Tạo DataFrame từ snippets của hàng hiện tại
            snippet_df = pd.DataFrame(snippets)
            
            # Kiểm tra xem DataFrame có các cột cần thiết không
            required_columns = ['offsetInBeginSection', 'offsetInEndSection', 'text', 'document', 'beginSection']
            missing_columns = set(required_columns) - set(snippet_df.columns)
            if missing_columns:
                for col in missing_columns:
                    snippet_df[col] = None

            # Xử lý cột document
            if 'document' in snippet_df.columns:
                snippet_df['document'] = snippet_df['document'].astype(str).str.split('/').str[-1]

            # Thêm các cột khác từ hàng gốc
            other_cols = pd.DataFrame(train_df.iloc[idx]).T
            snippet_df = pd.concat([snippet_df, other_cols], axis=1)

            # Điền tất cả giá trị NaN
            for column in snippet_df.columns:
                if column in ['text', 'offsetInBeginSection', 'offsetInEndSection']:
                    continue
                if snippet_df[column].isnull().any():
                    fill_value = snippet_df[column].iloc[0]
                    if isinstance(fill_value, list):
                        fill_value = [] if pd.isna(fill_value).any() else fill_value[0]
                    snippet_df[column] = snippet_df[column].fillna(fill_value)

            # Điền các giá trị cho question và ideal_answer
            snippet_df['question'] = snippet_df['question'].fillna(str(train_df['question'].iloc[idx]))
            # snippet_df['ideal_answer'] = snippet_df['ideal_answer'].fillna(str(train_df['ideal_answer'].iloc[idx]))
            snippet_df['extracted_answer'] = snippet_df['extracted_answer'].fillna(str(train_df['extracted_answer'].iloc[idx]))
            # Loại bỏ các hàng có cả 3 giá trị null
            null_mask = snippet_df[['offsetInBeginSection', 'offsetInEndSection', 'text']].isnull().all(axis=1)
            snippet_df = snippet_df[~null_mask]

            # Thêm vào list các DataFrame đã xử lý
            transformed_dfs.append(snippet_df)
            
        except Exception as e:
            print(f"Lỗi xử lý hàng {idx}: {str(e)}")
            continue

    # Kiểm tra nếu không có DataFrame nào được tạo
    if not transformed_dfs:
        return pd.DataFrame()

    # Ghép tất cả các DataFrame con
    result_df = pd.concat(transformed_dfs, axis=0, ignore_index=True)

    # Loại bỏ các cột không cần thiết
    if 'snippets' in result_df.columns:
        result_df = result_df.drop(columns=['snippets'])
    if 'documents' in result_df.columns:
        result_df = result_df.drop(columns=['documents'])

    return result_df


In [22]:
# Chuyển đổi toàn bộ DataFrame
# transformed_df = transform_snippets_to_rows(train_df)
transformed_df = transform_snippets_to_rows(temp_df)
transformed_df

,offsetInBeginSection,offsetInEndSection,text,beginSection,document,endSection,question,ideal_answer,extracted_answer
0,131.0,358.0,Hirschsprung disease (HSCR) is a multifactoria...,abstract,15829955,abstract,Is Hirschsprung disease a mendelian or a multi...,"[Coding sequence mutations in RET, GDNF, EDNRB...","Coding sequence mutations in RET, GDNF, EDNRB,..."
1,554.0,992.0,"In this study, we review the identification of...",abstract,15617541,abstract,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","Coding sequence mutations in RET, GDNF, EDNRB,..."
2,397.0,939.0,"Coding sequence mutations in e.g. RET, GDNF, E...",abstract,12239580,abstract,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","Coding sequence mutations in RET, GDNF, EDNRB,..."
3,941.0,1279.0,For almost all of the identified HSCR genes in...,abstract,12239580,abstract,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","Coding sequence mutations in RET, GDNF, EDNRB,..."
4,129.0,358.0,Hirschsprung disease (HSCR) is a multifactori...,abstract,15829955,abstract,Is Hirschsprung disease a mendelian or a multi...,"Coding sequence mutations in RET, GDNF, EDNRB,...","Coding sequence mutations in RET, GDNF, EDNRB,..."
...,...,...,...,...,...,...,...,...,...
60151,0.0,78.0,Glofitamab Treatment in Relapsed or Refractory...,title,35626120,title,What disease can be treated with Glofitamab?,NaN,Glofitamab is being tested for treatment of DL...
60152,361.0,624.0,"In this study, we evaluated the safety and eff...",abstract,35626120,abstract,What disease can be treated with Glofitamab?,NaN,Glofitamab is being tested for treatment of DL...
60153,1106.0,1284.0,Our data suggest that glofitamab treatment is ...,abstract,35626120,abstract,What disease can be treated with Glofitamab?,NaN,Glofitamab is being tested for treatment of DL...
60154,674.0,976.0,"Bispecific antibodies such as epcoritamab, mos...",abstract,36198538,abstract,What disease can be treated with Glofitamab?,NaN,Glofitamab is being tested for treatment of DL...


In [23]:
# Check if exist Nan valuea
transformed_df.isnull().sum()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


offsetInBeginSection        0
offsetInEndSection          0
text                        0
beginSection                0
document                    0
endSection                  0
question                    0
ideal_answer            60128
extracted_answer            0
dtype: int64

In [24]:
# Check if text is duplicated
transformed_df.duplicated(subset=['text']).sum()

2697

In [25]:
# Save data
transformed_df.to_csv("/kaggle/working/QA.csv")

# Fine-tuning model

In [21]:
# Split the data into training and validation sets
split_index = int(len(transformed_df) * 0.8)
train_df = transformed_df[:split_index]
val_df = transformed_df[split_index:]

In [22]:
# Choose a pre-trained model and tokenizer
tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/all-MiniLM-L6-v2')
model = AutoModelForQuestionAnswering.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Some weights of Qwen2ForQuestionAnswering were not initialized from the model checkpoint at Qwen/Qwen2.5-0.5B-Instruct and are newly initialized: ['embed_tokens.weight', 'layers.0.input_layernorm.weight', 'layers.0.mlp.down_proj.weight', 'layers.0.mlp.gate_proj.weight', 'layers.0.mlp.up_proj.weight', 'layers.0.post_attention_layernorm.weight', 'layers.0.self_attn.k_proj.bias', 'layers.0.self_attn.k_proj.weight', 'layers.0.self_attn.o_proj.weight', 'layers.0.self_attn.q_proj.bias', 'layers.0.self_attn.q_proj.weight', 'layers.0.self_attn.v_proj.bias', 'layers.0.self_attn.v_proj.weight', 'layers.1.input_layernorm.weight', 'layers.1.mlp.down_proj.weight', 'layers.1.mlp.gate_proj.weight', 'layers.1.mlp.up_proj.weight', 'layers.1.post_attention_layernorm.weight', 'layers.1.self_attn.k_proj.bias', 'layers.1.self_attn.k_proj.weight', 'layers.1.self_attn.o_proj.weight', 'layers.1.self_attn.q_proj.bias', 'layers.1.self_attn.q_proj.weight', 'layers.1.self_attn.v_proj.bias', 'layers.1.self_attn.v_

In [23]:
# Prepare the dataset using the tokenizer
train_encodings = tokenizer(train_df['text'].astype(str).tolist(), train_df['question'].astype(str).tolist(), truncation=True, padding=True )
val_encodings = tokenizer(val_df['text'].astype(str).tolist(), val_df['question'].astype(str).tolist(), truncation=True, padding=True)

In [24]:
def add_token_positions(encodings, df):
    """
    Thêm vị trí token start và end dựa trên offsetInBeginSection và offsetInEndSection

    Parameters:
    -----------
    encodings: transformers.BatchEncoding
        Kết quả mã hóa từ tokenizer
    df: DataFrame
        DataFrame chứa các cột offsetInBeginSection và offsetInEndSection
    """
    start_positions = []
    end_positions = []

    for i in range(len(encodings.input_ids)):
        # Lấy offset từ DataFrame
        if(df['offsetInBeginSection'][i] < 0):
            start_char = 0
        elif(df['offsetInEndSection'][i] < 0):
            end_char = 0
        else:
            start_char = int(df['offsetInBeginSection'].iloc[i])
            end_char = int(df['offsetInEndSection'].iloc[i])

        # Chuyển đổi từ vị trí ký tự sang vị trí token
        # token_start_idx là token đầu tiên chứa ký tự start_char
        # token_end_idx là token cuối cùng chứa ký tự end_char
        token_start_idx = encodings.char_to_token(i, start_char)
        token_end_idx = encodings.char_to_token(i, end_char)

        # Xử lý trường hợp không tìm thấy token
        if token_start_idx is None:
            token_start_idx = 0
        if token_end_idx is None:
            token_end_idx = 0

        # Đảm bảo end_idx không nhỏ hơn start_idx
        if token_end_idx < token_start_idx:
            token_end_idx = token_start_idx

        start_positions.append(token_start_idx)
        end_positions.append(token_end_idx)

    # Cập nhật encodings với vị trí start và end
    encodings.update({'start_positions': start_positions, 'end_positions': end_positions})


add_token_positions(train_encodings, train_df)
add_token_positions(val_encodings, train_df)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [25]:
# Create a custom dataset class
import torch
class QADataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings.input_ids)

train_dataset = QADataset(train_encodings)
val_dataset = QADataset(val_encodings)

In [26]:
# # Set up training arguments and the Trainer
# training_args = TrainingArguments(
#     output_dir='./kaggle/working/results',
#     run_name="experiment_1",  # Thêm tên run riêng biệt
#     num_train_epochs=11,
#     per_device_train_batch_size=10,
#     per_device_eval_batch_size=9,
#     logging_dir='./kaggle/working/logs',
#     learning_rate=1e-4, # Custom learning rate
# )

# Set up the training arguments
training_args = TrainingArguments(
    output_dir='./kaggle/working/results',
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=1e-4,
    per_device_train_batch_size=6,
    per_device_eval_batch_size=6,
    num_train_epochs=4,
    optim="paged_adamw_32bit",
    # fp16=True,
    weight_decay=0.01,
    push_to_hub=False,
    report_to="wandb",  # enable logging to W&B
    run_name="fine_tune_BioASQ",  # name of the W&B run (optional)
    logging_steps=1,  # how often to log to W&B
    logging_dir=os.path.join('./kaggle/working/results', "logs"),
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [27]:
# Set up the data collator

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    optimizers=(AdamW(model.parameters(), lr=1e-4), None), # Custom optimizer with the learning rate
)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
/usr/local/lib/python3.10/dist-packages/transformers/optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:494: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


In [ ]:
# Fine-tune the model
trainer.train()

<frozen importlib._bootstrap>:671: ImportWarning: DynamicImporter.exec_module() not found; falling back to load_module()
<frozen importlib._bootstrap>:671: ImportWarning: DynamicImporter.exec_module() not found; falling back to load_module()
/usr/local/lib/python3.10/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
/usr/local/lib/python3.10/dist-packages/pydantic/main.py:1309: PydanticDeprecatedSince20: The `copy` method is deprecated; use `model_copy` instead. See the docstring of `BaseModel.copy` for details about how to handle `include` and `exclude`. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.9/migration/
  warnings.warn(
wandb: Currently logged in as: uit-21522592 (21522798

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:68: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn('Was asked to gather along dimension 0, but all '


Epoch,Training Loss,Validation Loss


In [ ]:
model.save_pretrained("./kaggle/working//model.pt")

# Evaluate model

In [25]:
# Function to divide sentence
from nltk.tokenize import sent_tokenize
sent_tokenize(transformed_df['extracted_answer'][0])

/usr/local/lib/python3.10/dist-packages/nltk/decorators.py:69: DeprecationWarning: `formatargspec` is deprecated since Python 3.5. Use `signature` and the `Signature` object directly
  signature = inspect.formatargspec(regargs, varargs, varkwargs, defaults,


['Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease.',
 "The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model."]

In [26]:
print(transformed_df['question'][0])
sent_tokenize(transformed_df['question'][0])

Is Hirschsprung disease a mendelian or a multifactorial disorder?


['Is Hirschsprung disease a mendelian or a multifactorial disorder?']

In [27]:
import re
alphabets= "([A-Za-z])"
prefixes = "(Mr|St|Mrs|Ms|Dr)[.]"
suffixes = "(Inc|Ltd|Jr|Sr|Co)"
starters = "(Mr|Mrs|Ms|Dr|Prof|Capt|Cpt|Lt|He\\s|She\\s|It\\s|They\\s|Their\\s|Our\\s|We\\s|But\\s|However\\s|That\\s|This\\s|Wherever)"
acronyms = "([A-Z][.][A-Z][.](?:[A-Z][.])?)"
websites = "[.](com|net|org|io|gov|edu|me)"
digits = "([0-9])"
multiple_dots = r'\.{2,}'

def split_into_sentences(text: str) -> list[str]:
    """
    Split the text into sentences.

    If the text contains substrings "<prd>" or "<stop>", they would lead 
    to incorrect splitting because they are used as markers for splitting.

    :param text: text to be split into sentences
    :type text: str

    :return: list of sentences
    :rtype: list[str]
    """
    text = " " + text + "  "
    text = text.replace("\n"," ")
    text = re.sub(prefixes,"\\1<prd>",text)
    text = re.sub(websites,"<prd>\\1",text)
    text = re.sub(digits + "[.]" + digits,"\\1<prd>\\2",text)
    text = re.sub(multiple_dots, lambda match: "<prd>" * len(match.group(0)) + "<stop>", text)
    if "Ph.D" in text: text = text.replace("Ph.D.","Ph<prd>D<prd>")
    text = re.sub("\\s" + alphabets + "[.] "," \\1<prd> ",text)
    text = re.sub(acronyms+" "+starters,"\\1<stop> \\2",text)
    text = re.sub(alphabets + "[.]" + alphabets + "[.]" + alphabets + "[.]","\\1<prd>\\2<prd>\\3<prd>",text)
    text = re.sub(alphabets + "[.]" + alphabets + "[.]","\\1<prd>\\2<prd>",text)
    text = re.sub(" "+suffixes+"[.] "+starters," \\1<stop> \\2",text)
    text = re.sub(" "+suffixes+"[.]"," \\1<prd>",text)
    text = re.sub(" " + alphabets + "[.]"," \\1<prd>",text)
    if "”" in text: text = text.replace(".”","”.")
    if "\"" in text: text = text.replace(".\"","\".")
    if "!" in text: text = text.replace("!\"","\"!")
    if "?" in text: text = text.replace("?\"","\"?")
    text = text.replace(".",".<stop>")
    text = text.replace("?","?<stop>")
    text = text.replace("!","!<stop>")
    text = text.replace("<prd>",".")
    sentences = text.split("<stop>")
    sentences = [s.strip() for s in sentences]
    if sentences and not sentences[-1]: sentences = sentences[:-1]
    return sentences

In [28]:
print(split_into_sentences(transformed_df['extracted_answer'][0]))

['Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease.', "The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model."]


In [29]:
!pip --quiet install evaluate

/usr/lib/python3.10/pty.py:89: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 2.7 MB/s eta 0:00:00


In [30]:
from evaluate import load
bertscore = load("bertscore")
predictions = ["Is Hirschsprung disease a mendelian or a multifactorial disorder?"]
references = ["Coding sequence mutations in RET, GDNF, EDNRB, EDN3, and SOX10 are involved in the development of Hirschsprung disease.The majority of these genes was shown to be related to Mendelian syndromic forms of Hirschsprung's disease, whereas the non-Mendelian inheritance of sporadic non-syndromic Hirschsprung disease proved to be complex; involvement of multiple loci was demonstrated in a multiplicative model."]
results = bertscore.compute(predictions=predictions, references=references, model_type="distilbert-base-uncased")
print(results)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

{'precision': [0.8073920011520386], 'recall': [0.7295451164245605], 'f1': [0.7664970755577087], 'hashcode': 'distilbert-base-uncased_L5_no-idf_version=0.3.12(hug_trans=4.44.2)'}


In [31]:
# Call model from Hugging Face
from transformers import pipeline

generator = pipeline(model="HuggingFaceH4/zephyr-7b-beta", device = "cud")

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

model-00001-of-00008.safetensors:   0%|          | 0.00/1.89G [00:00<?, ?B/s]

model-00002-of-00008.safetensors:   0%|          | 0.00/1.95G [00:00<?, ?B/s]

model-00003-of-00008.safetensors:   0%|          | 0.00/1.98G [00:00<?, ?B/s]

...

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.80M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 16.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 2430 has 14.74 GiB memory in use. Of the allocated memory 14.59 GiB is allocated by PyTorch, and 20.14 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)

In [ ]:
hyde = generator([{"role": "user", "content": transformed_df['question'][0]}], do_sample=False, max_new_tokens = 200)

In [33]:
print(hyde)

[{'generated_text': [{'role': 'user', 'content': 'Is Hirschsprung disease a mendelian or a multifactorial disorder?'}, {'role': 'assistant', 'content': 'Hirschsprung disease (HD) is a multifactorial disorder. While genetic factors play a role in the development of HD, it is not a simple Mendelian disorder. Studies have identified several genes that may contribute to HD, but the exact genetic mechanisms are complex and involve both genetic and environmental factors. In some cases, HD may be inherited, but in most cases, it occurs sporadically without a family history. The exact cause of HD is still not fully understood.'}]}]
